In [ ]:
# group perpetually popular vs temporarily popular vs never popular 
# get mean, SD, outliers of all? objects
# find a value to split groups by

In [1]:
from popularity_queries import wiki_pageviews, wiki_scrape
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# to csvs
wiki_scrape("https://en.wikipedia.org/wiki/List_of_galaxies", 1, table_ind = 2).to_csv("data/galaxies_list.csv", index=False)


In [45]:

import requests
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://en.wikipedia.org/wiki/List_of_minor_planets:_1%E2%80%931000"

response = requests.get(
    url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30
)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

results = {}

for table in soup.select("table.wikitable"):
    for row in table.select("tr"):
        cells = row.find_all("td", recursive=False)
        if not cells:
            continue

        # First column contains a link like "1 Ceres".
        link = cells[0].find("a", href=True)
        if not link:
            continue

        designation = link.get_text(" ", strip=True)
        match = re.match(r"^(\d+)\s+(.+)$", designation)

        if not match:
            continue

        number = int(match.group(1))
        name = match.group(2).strip()

        if 1 <= number <= 1000:
            results[number] = (
                name,
                urljoin("https://en.wikipedia.org", link["href"])
            )

output_file = "data/asteroids_list.csv"

with open(output_file, "w", encoding="utf-8") as f:
    for number in sorted(results):
        name, link = results[number]
        f.write(f"\"('{number} {name}', '{link}')\"\n")



In [46]:
def parse(data):
    return pd.read_csv(f"data/{data}.csv")['0'].apply(lambda x: x.split(',')[1].split("wiki/")[-1][:-2].split("_")).apply(lambda x: " ".join(x)).to_list()

In [47]:
# stars, galaxies, exoplanets, asteroids
stars = parse("stars_list")
galaxies = parse("galaxies_list")
exoplanets = parse("exoplanet_list")
nebulas = parse("nebula_list")
asteroids = parse("asteroids_list")
everything = stars + galaxies + exoplanets + nebulas + asteroids

In [101]:
stars_pv = wiki_pageviews(stars)
galaxies_pv = wiki_pageviews(galaxies)
exoplanets_pv = wiki_pageviews(exoplanets)
asteroids_pv = wiki_pageviews(asteroids)
nebulas_pv = wiki_pageviews(nebulas)

Current article: 1392
Current article: 68
Current article: 1349
Current article: 999
Current article: 110


In [102]:
everything_pv = pd.concat([stars_pv, galaxies_pv, exoplanets_pv, asteroids_pv, nebulas_pv], axis=1)

In [146]:
# Configuration
INPUT_FILE = "daily_pageviews.csv"
OUTPUT_FILE = "celestial_popularity_classified.csv"

BASELINE_PERCENTILE = 98
# 97% -> 65
# 98% -> 125
# 99% -> 285
CONSISTENCY_THRESHOLD = 0.75
SPIKE_RATIO = 10
MIN_SPIKE_DAYS = 3

ROLLING_BASELINE_DAYS = 28
MAD_MULTIPLIER = 6
RECOVERY_RATIO = 2
RECOVERY_WINDOW_DAYS = 14
RECOVERY_CONSECUTIVE_AVERAGES = 3

In [160]:
# set index to datetime
df = everything_pv.copy().reset_index()
df.rename(columns={'index': 'date'}, inplace=True)
date_col = df.columns[0]
df[date_col] = pd.to_datetime(df[date_col],format="%Y%m%d%H",errors="coerce")
df = df.set_index(date_col)
df = df.loc[:, ~df.columns.duplicated()]


In [161]:
# get median views for each object -> get a threshold for 'perpetually popular'
baselines = df.median(axis=0, skipna=True)
global_threshold = np.percentile(baselines, BASELINE_PERCENTILE)
print("Global high-view threshold:", global_threshold)

Global high-view threshold: 128.01999999999998


In [147]:
def consecutive_runs(mask):
    mask = np.asarray(mask, dtype=bool)
    edges = np.diff(
        np.r_[False, mask, False].astype(int)
    )
    starts = np.where(edges == 1)[0]
    ends = np.where(edges == -1)[0] - 1
    return list(zip(starts, ends))

In [173]:
# analyze
output = pd.DataFrame()
results = []
for obj in df.columns:
    views = df[obj]
    baseline = baselines[obj]

    # percentage of columns w view counts greater than threshold
    consistency = (views >= global_threshold).mean()

    # rolling 3d avg
    rolling_3d = views.rolling(window=3, min_periods=3).mean()
    # rolling baseline from 3 days ago (prevent spike from influencing its own baseline)
    local_median = views.rolling(window=ROLLING_BASELINE_DAYS,min_periods=14).median().shift(3)

    # get median absolute deviation - how much the view counts have fluctuated over rolling window
    deviations = (views - local_median).abs()
    local_mad = deviations.rolling( window=ROLLING_BASELINE_DAYS,min_periods=14).median()
    robust_sd = (1.4826 * local_mad).clip(lower=1)
 
    # prevents division by zero.
    reference_baseline = max(float(baseline), 1.0)

    # is the spike large relative to the object's baseline?
    ratio_condition = (rolling_3d >= SPIKE_RATIO * reference_baseline)    
    # is the spike unusual relative to recent fluctuations?
    robust_condition = (rolling_3d >local_median + MAD_MULTIPLIER * robust_sd)
    # -> both conditions must pass for it to be considered a 'spike'

    # check for spikes lasting a certain length of time
    potential_spike = (ratio_condition & robust_condition).fillna(False)
    spikes = [(start, end) for start, end in consecutive_runs(potential_spike) if end - start + 1 >= MIN_SPIKE_DAYS]

    # check if spike is temporary or not. recovered means it went back to somewhat normal
    detected_spike = bool(spikes)
    recovered_spike = False
    spike_start = pd.NaT
    spike_end = pd.NaT
    spike_ratio = np.nan
    largest_spike_ratio = -np.inf
    largest_spike_start = pd.NaT
    largest_spike_end = pd.NaT

    for start, end in spikes:
        peak_average = rolling_3d.iloc[start:end + 1].max()
        current_spike_ratio = peak_average / reference_baseline

        # keep track of the largest detected spike.
        if current_spike_ratio > largest_spike_ratio:
            largest_spike_ratio = current_spike_ratio
            largest_spike_start = views.index[start]
            largest_spike_end = views.index[end]

        future = rolling_3d.iloc[end + 1:end + 1 + RECOVERY_WINDOW_DAYS]

        # Recovery requires 3 consecutive rolling averages
        # at or below twice the baseline.
        recovery_condition = (future <= RECOVERY_RATIO * reference_baseline).fillna(False)

        sustained_recovery = recovery_condition.rolling(window=RECOVERY_CONSECUTIVE_AVERAGES, min_periods=RECOVERY_CONSECUTIVE_AVERAGES).sum().ge(RECOVERY_CONSECUTIVE_AVERAGES).any()

        if sustained_recovery:
            recovered_spike = True
            if current_spike_ratio > spike_ratio:
                spike_start = views.index[start]
                spike_end = views.index[end]
                spike_ratio = current_spike_ratio

    if not recovered_spike and detected_spike:
        spike_start = largest_spike_start
        spike_end = largest_spike_end
        spike_ratio = largest_spike_ratio

    
    metrics = {"median_daily_views": baseline, "consistency_fraction": consistency, "spike_ratio": spike_ratio, "detected_spike_candidate": detected_spike, "qualifying_recovered_spike": recovered_spike}
    
    high_baseline = baseline >= global_threshold
    consistent = (metrics["consistency_fraction"]>= CONSISTENCY_THRESHOLD)

    if high_baseline and consistent:
        category = "Perpetually popular"

    elif metrics["qualifying_recovered_spike"]:
        category = "Temporarily popular"

    elif metrics["detected_spike_candidate"]:
        category = "Unclassified"

    elif not high_baseline:
        category = "Never popular"

    else:
        category = "Unclassified"

    results.append({"object": obj, "category": category, "global_threshold": global_threshold, **metrics,})

output = pd.DataFrame(results)

In [178]:
output[output['category'] == 'Perpetually popular'].reset_index()

,index,object,category,global_threshold,median_daily_views,consistency_fraction,spike_ratio,detected_spike_candidate,qualifying_recovered_spike
0,0,Pollux (star),Perpetually popular,128.02,212.0,0.989047,NaN,False,False
1,1,Arcturus,Perpetually popular,128.02,780.0,1.000000,NaN,False,False
2,2,Capella,Perpetually popular,128.02,300.0,1.000000,NaN,False,False
3,13,Aldebaran,Perpetually popular,128.02,516.0,1.000000,NaN,False,False
4,34,Lists of stars,Perpetually popular,128.02,261.0,0.999189,NaN,False,False
5,35,Star formation,Perpetually popular,128.02,345.0,1.000000,NaN,True,True
6,36,Stellar evolution,Perpetually popular,128.02,703.0,1.000000,NaN,False,False
7,37,Stellar classification,Perpetually popular,128.02,1151.0,1.000000,NaN,False,False
8,41,Stellar nucleosynthesis,Perpetually popular,128.02,431.0,1.000000,NaN,False,False
9,43,Star system,Perpetually popular,128.02,350.0,1.000000,NaN,False,False
